# tSCS EMG — single pulse — polarity × lidocaine (subject NTA, 24-07-2026)

One participant, one session, a **2 × 2 design**: stimulation polarity (**cathodic** = polarity 2,
the usual cathode-on-the-spine montage; **anodic** = polarity 1) × lidocaine (**before** / **with**).
Electrode 2, anode at the iliac crests, folder `testSCS` (`NTA_2026-07-24_polarity_lidocaine.xlsx`).



> **Run §2 first.** The latencies are picked by hand, once per recording, and saved to
> `results/`. Until a recording has saved picks, the sections below raise
> `FileNotFoundError: No saved latencies for ...` for that file — that is the notebook asking
> you to go and pick them, not a failure of the analysis.

## What is measured
- **Latency** — picked **by hand** per muscle × intensity (§2, once per file; saved to
  `results/latency_<file>.csv`). `NaN` = no response.
- **Peak-to-peak** — max − min in `[latency + OFFSET_MS, + WINDOW_MS]` (mV).

## Colours and styles, everywhere
**gray = before lidocaine, orange = with lidocaine** — the same two colours as in the original
notebooks. **Polarity is the style: cathodic = solid lines / plain bars / filled markers,
anodic = dashed lines / hatched bars / hollow markers.** So colour answers "lidocaine?", style
answers "which polarity?".

## Files
| | single pulse | burst | ARC-EX (Modulated) |
|---|---|---|---|
| **cathodic · before** | `100913` (10:09) | `102443` (10:24; `102236` aborted) | `103530` (10:35; `103040` no response) |
| **cathodic · lidocaine** | `113447` (11:34) | `113834` (11:38) | `114332` (11:43) |
| **anodic · before** | `101941` (10:19) | `102721` (10:27) | `103849` (10:38) |
| **anodic · lidocaine** | `113632` (11:36) | `114055` (11:40) | `114730` (11:47; `114630` ignore) |

Motor threshold from the log — burst: cathodic 25 mA before / 30 lidocaine, anodic 30 / 30;
ARC-EX: cathodic 70 / 70, anodic 65 / 90; single pulse ~30–40. Participant: *"less pain with
anodic"*, *"much less pain after lidocaine"*. Lidocaine applied 45 min (~10:50 → 11:33).

> **Intensity rule:** every muscle is analysed **at the motor threshold picked for it** in
> `notebooks/motor_thresholds/` - the trace you chose there is the trace the numbers come from.
> Never one mA for the whole arm, never a step above. This notebook's `AMP_MT` is only the single
> sweep the *diagnostic* figures inspect; it is derived from those picks, not typed from the log.


In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall, waterfall_overlay,
                       latency_picker, save_latency_csv, load_latency_csv)
from functions.quantify import plot_p2p_markers
from functions.average import compare_per_muscle
set_style()


## 1 · Config

**What this does.** Names the recordings, the conditions compared, and the peak-detection settings
(shared with every other notebook - change them here and they no longer agree). Prints each
recording's intensity ladder and where its thresholds came from.


In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
COND = {   # label: (file,)
    "cathodic · before":    ("Single_Pulse_autosave_20260724_100913_756ms.csv",),
    "cathodic · lidocaine": ("Single_Pulse_autosave_20260724_113447_649ms.csv",),
    "anodic · before":      ("Single_Pulse_autosave_20260724_101941_811ms.csv",),
    "anodic · lidocaine":   ("Single_Pulse_autosave_20260724_113632_480ms.csv",),
}
COLOURS = {"cathodic · before": "0.45", "cathodic · lidocaine": "#f39c12",     # colour = lidocaine
           "anodic · before": "0.45", "anodic · lidocaine": "#f39c12"}
STYLE = {   # style = polarity: (bar hatch, line style, marker) - here only 'hollow' is used
    "cathodic": ("",    "-",  "o"),
    "anodic":   ("///", "--", "s"),
}
def style(keys, what):    # what: "hatch" | "ls" | "marker" | "hollow"
    i = {"hatch": 0, "ls": 1, "marker": 2}.get(what)
    return [STYLE[k.split(" · ")[0]][i] if what != "hollow" else k.startswith("anodic") for k in keys]

XLIM      = (-20, 80)   # ms shown in the waterfalls / picker
WINDOW_MS = 20.0        # peak-to-peak window length (ms)
OFFSET_MS = 2.0         # ...starting this long after the picked latency (ms)
YLIM_LAT  = (0, 30)     # y-range for latency plots (ms)

LABELS = list(COND)
CSVS   = [D + COND[k][0] for k in LABELS]
COLS   = [COLOURS[k] for k in LABELS]
RUNS   = [load_run(f) for f in CSVS]
muscles = [c for c in RUNS[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS)]

def _latfile(csv):
    from functions.io import result_path        # results/<participant-session>/latency_*.csv
    return result_path(csv, "latency")
def sel(*keys):
    return ([D + COND[k][0] for k in keys], list(keys), [COLOURS[k] for k in keys])
HOLLOW = style(LABELS, "hollow")
for lab, csv, (meta, t, sig) in zip(LABELS, CSVS, RUNS):
    print(f"{lab:22s} {[m['amp_ma'] for m in meta]} mA | " + ("picks saved" if os.path.exists(_latfile(csv)) else "NO PICKS YET"))


## 2 · Manual latency picking — one block per file, do each ONCE

Each block: **A** loads the file (safe to re-run), **B** the picker (uncomment, run, click the
onset for every muscle × intensity — *Set NaN* = no response — then comment again), **C** saves to
that file's own CSV (uncomment, run, comment again). Blocks only talk to each other, and a save is
refused if the picks don't match the file's intensities. Skip once every file says *picks saved*.

**What this does.** The single-pulse pipeline does not detect peaks automatically: you click the
response onset for every muscle x intensity, once per recording, and it is saved to `results/`.
`NaN` = no response. Do it once and comment the cells again.


### 2·cathodic · before

In [ ]:
# ---- cathodic · before · A: load ----
CSV_cat_b = D + COND["cathodic · before"][0]
meta_cat_b, t_cat_b, sig_cat_b = load_run(CSV_cat_b)
picks_cat_b = load_latency_csv(_latfile(CSV_cat_b)) if os.path.exists(_latfile(CSV_cat_b)) else None
print("cathodic · before:", CSV_cat_b.split("/")[-1], "|", [m["amp_ma"] for m in meta_cat_b], "mA")
print("existing picks reloaded - continue/correct them" if picks_cat_b else "no picks yet - pick them in B")


In [ ]:
# ---- cathodic · before · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_cat_b = latency_picker(meta_cat_b, t_cat_b, sig_cat_b, muscles, xlim=XLIM, manual_peaks=picks_cat_b)


In [ ]:
# ---- cathodic · before · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_cat_b, muscles, CSV_cat_b, meta=meta_cat_b))


### 2·cathodic · lidocaine

In [ ]:
# ---- cathodic · lidocaine · A: load ----
CSV_cat_l = D + COND["cathodic · lidocaine"][0]
meta_cat_l, t_cat_l, sig_cat_l = load_run(CSV_cat_l)
picks_cat_l = load_latency_csv(_latfile(CSV_cat_l)) if os.path.exists(_latfile(CSV_cat_l)) else None
print("cathodic · lidocaine:", CSV_cat_l.split("/")[-1], "|", [m["amp_ma"] for m in meta_cat_l], "mA")
print("existing picks reloaded - continue/correct them" if picks_cat_l else "no picks yet - pick them in B")


In [ ]:
# ---- cathodic · lidocaine · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_cat_l = latency_picker(meta_cat_l, t_cat_l, sig_cat_l, muscles, xlim=XLIM, manual_peaks=picks_cat_l)


In [ ]:
# ---- cathodic · lidocaine · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_cat_l, muscles, CSV_cat_l, meta=meta_cat_l))


### 2·anodic · before

In [ ]:
# ---- anodic · before · A: load ----
CSV_an_b = D + COND["anodic · before"][0]
meta_an_b, t_an_b, sig_an_b = load_run(CSV_an_b)
picks_an_b = load_latency_csv(_latfile(CSV_an_b)) if os.path.exists(_latfile(CSV_an_b)) else None
print("anodic · before:", CSV_an_b.split("/")[-1], "|", [m["amp_ma"] for m in meta_an_b], "mA")
print("existing picks reloaded - continue/correct them" if picks_an_b else "no picks yet - pick them in B")


In [ ]:
# ---- anodic · before · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_an_b = latency_picker(meta_an_b, t_an_b, sig_an_b, muscles, xlim=XLIM, manual_peaks=picks_an_b)


In [ ]:
# ---- anodic · before · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_an_b, muscles, CSV_an_b, meta=meta_an_b))


### 2·anodic · lidocaine

In [ ]:
# ---- anodic · lidocaine · A: load ----
CSV_an_l = D + COND["anodic · lidocaine"][0]
meta_an_l, t_an_l, sig_an_l = load_run(CSV_an_l)
picks_an_l = load_latency_csv(_latfile(CSV_an_l)) if os.path.exists(_latfile(CSV_an_l)) else None
print("anodic · lidocaine:", CSV_an_l.split("/")[-1], "|", [m["amp_ma"] for m in meta_an_l], "mA")
print("existing picks reloaded - continue/correct them" if picks_an_l else "no picks yet - pick them in B")


In [ ]:
# ---- anodic · lidocaine · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_an_l = latency_picker(meta_an_l, t_an_l, sig_an_l, muscles, xlim=XLIM, manual_peaks=picks_an_l)


In [ ]:
# ---- anodic · lidocaine · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_an_l, muscles, CSV_an_l, meta=meta_an_l))


## 3 · Raw traces — waterfalls

Per polarity: before, with lidocaine (same gain per muscle), and the two overlaid.

**What this does.** Every intensity stacked at its own mA, all conditions on a shared gain.
The overview of the sweep before any number is taken from it.


In [ ]:
for pol in ("cathodic", "anodic"):
    files, labs, cols = sel(f"{pol} · before", f"{pol} · lidocaine")
    runs = [load_run(f) for f in files]
    print(f"===== {labs[0]}"); g = waterfall(*runs[0], muscles, xlim=XLIM)
    print(f"===== {labs[1]}"); waterfall(*runs[1], muscles, xlim=XLIM, gains=g)
    print(f"===== overlay"); waterfall_overlay(runs, muscles=muscles, xlim=XLIM, gains=g, labels=labs, colours=cols)


## 4 · Check — which peaks the peak-to-peak uses (files without picks are skipped)

**What this does.** Shows which peak the peak-to-peak is measured from, given the latency you
picked. Recordings without saved picks are skipped rather than guessed at.


In [ ]:
for lab, csv, (meta, t, sig) in zip(LABELS, CSVS, RUNS):
    if not os.path.exists(_latfile(csv)):
        print(f"{lab}: no picks yet - skipped"); continue
    print("=====", lab)
    plot_p2p_markers(meta, t, sig, muscles, load_latency_csv(_latfile(csv)), window_ms=WINDOW_MS, offset_ms=OFFSET_MS)


## 5 · Latency and peak-to-peak per muscle

One panel per muscle, x = intensity; ● solid = left arm, ▲ dashed = right arm; filled = cathodic,
hollow = anodic.
**5a** lidocaine within each polarity · **5b** polarity before and with lidocaine · **5c** all four.
Needs the picks of the files involved.

**What this does.** Latency and peak-to-peak per muscle, across conditions. Latency is the one
measure that needs no normalising between people - milliseconds are milliseconds.


In [ ]:
for pol in ("cathodic", "anodic"):                                   # 5a
    files, labs, cols = sel(f"{pol} · before", f"{pol} · lidocaine")
    compare_per_muscle(files, None, metric="latency", ylim=YLIM_LAT, labels=labs, colours=cols, hollow=style(labs, "hollow"))
    compare_per_muscle(files, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=labs, colours=cols, hollow=style(labs, "hollow"))


In [ ]:
for state in ("before", "lidocaine"):                                # 5b
    files, labs, cols = sel(f"cathodic · {state}", f"anodic · {state}")
    compare_per_muscle(files, None, metric="latency", ylim=YLIM_LAT, labels=labs, colours=cols, hollow=style(labs, "hollow"))
    compare_per_muscle(files, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=labs, colours=cols, hollow=style(labs, "hollow"))


In [ ]:
compare_per_muscle(CSVS, None, metric="latency", ylim=YLIM_LAT, labels=LABELS, colours=COLS, hollow=HOLLOW)      # 5c
compare_per_muscle(CSVS, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=LABELS, colours=COLS, hollow=HOLLOW);
